# Building ML Pipelines

Up to this point in the curriculum, we have been working with perfectly clean, synthetic matrices. Every column was a number, and there were no missing values. 

In the enterprise world, raw data is an absolute disaster. A database export will contain missing values (`NaN`), text categories (`"New York"`, `"London"`), and numeric values on wildly different scales. Before an algorithm can process this data, it must be cleansed, imputed, scaled, and encoded. 

If you do this manually, you will almost certainly introduce **Data Leakage**, corrupting your model. In this lesson, we will learn how to build robust, leak-proof automated assembly lines using `scikit-learn`'s **Pipeline** and **ColumnTransformer**.

A Pipeline is a sequence of data processing components. It mathematically guarantees that every transformation applied to your training data is applied perfectly and consistently to your test data (and future production data), without cross-contaminating the sets.

Let's set up our Python environment to build an industrial-grade ML workflow.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("✅ MLOps Pipeline Engineering Environment Ready.")

✅ MLOps Pipeline Engineering Environment Ready.


# 1. The Anatomy of Preprocessing (Transformers)

In `scikit-learn`, objects that interact with data are divided into two distinct mathematical categories:

1.  **Estimators (Models)**: Algorithms like `RandomForest` or `LogisticRegression`. They learn from data and make predictions. They have the methods `.fit()` and `.predict()`.
2.  **Transformers (Filters)**: Algorithms that modify the data itself. They have the methods `.fit()` (to learn the parameters of the transformation) and `.transform()` (to actually apply the math). 

Common enterprise transformers include:
* **Imputer**: Finds missing values (`NaN`) and replaces them with the mean, median, or a constant.
* **Scaler**: Standardizes numeric data using the Z-score formula $z = \frac{x - \mu}{\sigma}$.
* **One-Hot Encoder**: Converts text categories (like "Red", "Blue") into binary columns ($0$ or $1$) because algorithms cannot multiply text.

# 2. The Danger of Manual Preprocessing (Data Leakage)

Imagine you are standardizing your dataset using a `StandardScaler`. To do this, the Scaler must calculate the Mean ($\mu$) and Standard Deviation ($\sigma$) of your columns.

**The Catastrophic Error:**

```python
# ❌ DO NOT DO THIS 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_entire_dataset) # LEAKAGE!
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)
```
If you run `.fit_transform()` on the *entire* dataset before splitting it, the Mean and Variance of the **Test Set** have mathematically "leaked" into the Scaler. Your training data has been modified using knowledge from the future. Your Cross-Validation scores are now completely fraudulent.

**The Proper Mathematical Workflow:**
1. Split the data *first*.
2. `.fit()` the Scaler **strictly on the Training Set** (learn $\mu$ and $\sigma$ from the past).
3. `.transform()` the Training Set.
4. `.transform()` the Test Set using the exact parameters learned from Step 2.

Manually typing this out for 5 different preprocessing steps across K-Fold Cross Validation is a nightmare. This is exactly what the `Pipeline` automates.

# 3. The ColumnTransformer: Branching Logic

Enterprise datasets are heterogeneous. You cannot pass a column of City Names into a `StandardScaler` (it will crash trying to find the mean of a word). You cannot pass a column of Annual Income into a `OneHotEncoder` (it will create 10,000 useless binary columns).

We must split the data streams using a `ColumnTransformer`. This object allows us to apply a specific sequence of transformers to numeric columns, and a completely different sequence to categorical columns, before seamlessly merging them back together into a single matrix.

# 4. Building the Ultimate ML Pipeline in Code

Let's simulate a messy corporate dataset (Customer Churn) with missing numeric data, missing text data, and raw unscaled numbers. We will build a unified Pipeline to handle it all automatically.

In [4]:
# 1. Simulate Messy Enterprise Data
np.random.seed(42)
data = pd.DataFrame({
    'Age': np.random.choice([25, 34, 45, 50, np.nan], size=1000),             # Numeric with missing
    'Monthly_Spend': np.random.uniform(20, 200, size=1000),                   # Numeric continuous
    'Region': np.random.choice(['North', 'South', 'East', np.nan], size=1000),# Categorical with missing
    'Churn': np.random.randint(0, 2, size=1000)                               # Target
})

# 2. Separate Features and Target, then Split IMMEDIATELY
X = data.drop('Churn', axis=1)
y = data['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Define the Column Groups
numeric_features = ['Age', 'Monthly_Spend']
categorical_features = ['Region']

# 4. Build the Numeric Processing Stream
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Step 1: Fill missing numbers with the median
    ('scaler', StandardScaler())                   # Step 2: Scale the numbers
])

# 5. Build the Categorical Processing Stream
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), # Step 1: Fill missing text
    ('onehot', OneHotEncoder(handle_unknown='ignore'))                     # Step 2: Convert to binary columns
])

# 6. Combine Streams using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 7. Build the Final Unified Pipeline (Preprocessor + Algorithm)
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("🚀 Industrial Pipeline Constructed. Training...")

# 8. Train the entire Pipeline with a single command!
# The Pipeline automatically routes the data, fits the imputers/scalers strictly on X_train, 
# transforms X_train, and feeds the cleansed matrix to the Random Forest.
final_pipeline.fit(X_train, y_train)

# 9. Evaluate seamlessly
# The Pipeline automatically takes the raw X_test, pushes it through the pre-fitted imputers/scalers,
# and generates the predictions without a single drop of leakage.
y_pred = final_pipeline.predict(X_test)
print(f"🚨 Pipeline Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.1f}%")

🚀 Industrial Pipeline Constructed. Training...
🚨 Pipeline Test Accuracy: 47.5%


### The Magic of the Pipeline
If you deploy this model to a cloud server tomorrow, and the frontend app sends the model a raw JSON payload representing a new customer with a missing `Age` value, you do not need to write error-handling code. You simply pass the raw JSON straight into `final_pipeline.predict()`. The pipeline knows exactly how to impute, scale, encode, and score it automatically.

## Real-World Use Case or Analogy:
Think of an ML Pipeline like a **Car Manufacturing Assembly Line**:

* **Raw Materials ($X$)**: Scraps of metal, spools of wire, buckets of paint.
* **The Branches (`ColumnTransformer`)**: You cannot send rubber tires to the metal-stamping machine. The tires go down the 'Rubber Stream' (Categorical), and the steel goes down the 'Metal Stream' (Numeric).
* **The Transformations (`Transformers`)**: The steel is washed (Imputed) and cut (Scaled). The rubber is vulcanized (One-Hot Encoded).
* **The Assembly (`Estimator`)**: The cleanly processed parts arrive perfectly synchronized at the end of the belt, where the robotic arm (`RandomForest`) combines them into a finished car (a Prediction).
* **Leakage**: If you measure the size of the car *before* cutting the metal, your blueprints are ruined. Measurements (fitting) must happen inside the isolated stations.

---